Test of the entire mentions' extractor: (fake labels)

In [ ]:
import mentions_extractor_runner

runner = mentions_extractor_runner.MentionsExtractorRunner()

runner.train()

Parsing one document:

In [2]:
import utils
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
tokenized_text, word_ids_to_num_bert_tokens, entities = utils.get_conll_data("11_alices_adventures_in_wonderland_brat.conll", tokenizer)
print(entities)

[[3, 4, 0], [5, 5, 1], [15, 15, 1], [15, 16, 2], [18, 19, 3], [31, 31, 1], [37, 37, 1], [37, 38, 2], [64, 64, 1], [73, 73, 1], [77, 77, 1], [84, 84, 1], [92, 92, 1], [122, 127, 4], [131, 131, 1], [144, 144, 1], [156, 157, 4], [169, 169, 4], [177, 177, 1], [186, 186, 1], [188, 188, 1], [209, 210, 4], [230, 230, 1], [233, 233, 1], [240, 240, 1], [243, 243, 1], [248, 262, 5], [269, 269, 1], [272, 273, 6], [275, 275, 4], [285, 285, 4], [288, 293, 0], [300, 300, 1], [302, 302, 4], [309, 310, 7], [311, 311, 1], [318, 319, 0], [324, 325, 8], [339, 339, 1], [348, 348, 1], [350, 350, 1], [352, 352, 1], [355, 358, 9], [361, 362, 9], [368, 368, 1], [374, 374, 1], [380, 380, 1], [386, 386, 1], [399, 399, 1], [408, 408, 1], [423, 423, 1], [429, 430, 9], [446, 446, 1], [455, 455, 1], [466, 466, 1], [479, 479, 1], [486, 486, 1], [498, 498, 1], [511, 511, 1], [521, 521, 1], [523, 523, 1], [533, 533, 1], [540, 540, 1], [544, 544, 1], [548, 548, 1], [550, 550, 1], [554, 554, 1], [564, 564, 1], [570, 571

Getting all training data

In [ ]:
import utils
import torch
from transformers import AutoTokenizer, AutoModel
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
encoder = AutoModel.from_pretrained("SpanBERT/spanbert-base-cased").to(device)
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
data_dict = utils.prepare_data(tokenizer, encoder)  # {doc_id_0: {encoding, true_entites}, doc_id_1: {encoding, true_entities}, ...}
print(data_dict.keys())

Test for getting labels

In [24]:
import torch
starting_index = torch.tensor([1,1,3,5])
ending_index = torch.tensor([2,3,4,6])
predicted_indices = torch.stack([starting_index, ending_index], dim=1)
print(starting_index.size())

true_entities = torch.tensor([[1, 2, 0], [1, 3, 3], [3, 5, 1], [5, 6, 0], [7, 8, 2]])
print((true_entities.size()))
true_indices = true_entities[:, :2]
print(true_indices.size())

predicted_expand = predicted_indices.unsqueeze(1).expand(-1, true_indices.shape[0], predicted_indices.shape[1])
true_expand = true_indices.unsqueeze(0).expand(predicted_indices.shape[0], -1, predicted_indices.shape[1])
print(predicted_expand)
print(true_expand)

mask = (predicted_expand == true_expand).all(-1).any(-1).short()
print(mask)

torch.Size([4])
torch.Size([5, 3])
torch.Size([5, 2])
tensor([[[1, 2],
         [1, 2],
         [1, 2],
         [1, 2],
         [1, 2]],

        [[1, 3],
         [1, 3],
         [1, 3],
         [1, 3],
         [1, 3]],

        [[3, 4],
         [3, 4],
         [3, 4],
         [3, 4],
         [3, 4]],

        [[5, 6],
         [5, 6],
         [5, 6],
         [5, 6],
         [5, 6]]])
tensor([[[1, 2],
         [1, 3],
         [3, 5],
         [5, 6],
         [7, 8]],

        [[1, 2],
         [1, 3],
         [3, 5],
         [5, 6],
         [7, 8]],

        [[1, 2],
         [1, 3],
         [3, 5],
         [5, 6],
         [7, 8]],

        [[1, 2],
         [1, 3],
         [3, 5],
         [5, 6],
         [7, 8]]])
tensor([1, 1, 0, 1], dtype=torch.int16)


In [1]:
import mentions_extractor_runner
runner = mentions_extractor_runner.MentionsExtractorRunner()
runner.train()

Some weights of BertModel were not initialized from the model checkpoint at SpanBERT/spanbert-base-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Processing: 6593_history_of_tom_jones_a_foundling_brat.conll
Processing: 271_black_beauty_brat.conll
Processing: 730_oliver_twist_brat.conll
Processing: 16357_mary_a_fiction_brat.conll
Processing: 2807_to_have_and_to_hold_brat.conll
Processing: 1023_bleak_house_brat.conll
Processing: 1155_the_secret_adversary_brat.conll
Processing: 238_dear_enemy_brat.conll
Processing: 1661_the_adventures_of_sherlock_holmes_brat.conll
Processing: 711_allan_quatermain_brat.conll
EPOCH 1:


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.48 GiB (GPU 0; 4.00 GiB total capacity; 2.16 GiB already allocated; 283.60 MiB free; 2.21 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF